# Testing the blurryness of keras layers transforms vs keras ImageDataGenerator transforms

Create sample images for sequentially applied keras layers transforms and the corresponding ImageDataGenerator transforms, and compare the results. The blurryness of the keras layers transforms may be due to stacked interpolation between each transform, as opposed to the ImageDataGenerator which applies all transforms in one go, and only performs interpolation once. The blurryness is more noticeable with the RandomZoom layer, which may be due to the fact that it performs a larger transformation than the other layers, and therefore requires more interpolation.

In [3]:
# libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# data
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

We will now create a sequential model for each of the three data augmentation techniques (translation, flip, zoom) using keras layers, and apply them sequentially to a sample image from the CIFAR-10 dataset. We will then create a corresponding ImageDataGenerator for each sequence of techniques and compare the two results. We will use the same random seed for both methods to ensure that the same transformations are applied, and we will use the same interpolation method (bilinear) for both methods to ensure a fair comparison. We will also use the same fill mode (nearest) for both methods to ensure that the same strategy is used for filling in pixels outside the boundaries of the input image.

In [6]:
# keras sequential
interpolation_method = 'bilinear' # Use bilinear interpolation for better quality data_aug_GPU_combined = keras.Sequential([ layers.RandomTranslation(0.1, 0.1, fill_mode="nearest", interpolation= interpolation_method, seed = 128), layers.RandomFlip("horizontal", seed = 128), layers.RandomZoom(0.1, fill_mode="nearest", interpolation= interpolation_method,

# rotation layer
keras_layer_rotation = keras.Sequential([
    layers.RandomRotation(1/36, # Rotate images by up to 10 degrees (1/36 of a full rotation)
                          fill_mode="nearest",
                          interpolation= interpolation_method,
                          seed = 128)
    ])

# translation layer
keras_layer_translation = keras.Sequential([
    layers.RandomTranslation(0.1,
                             0.1,
                             fill_mode="nearest",
                             interpolation= interpolation_method,
                             seed = 128)
    ])

# flip layer
keras_layer_flip = keras.Sequential([
    layers.RandomFlip("horizontal",
                      seed = 128)
    ])

# zoom layer
keras_layer_zoom = keras.Sequential([
    layers.RandomZoom(0.1,
                      fill_mode="nearest",
                      interpolation= interpolation_method,
                      seed = 128)
    ])

# CPU augmentation using ImageDataGenerator

# rotation
data_aug_CPU_rotation = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=10,            # Rotate images by up to 10 degrees
    fill_mode='nearest',         # Fill strategy for pixels outside boundaries
    interpolation_order=1,       # Use bilinear interpolation (order=1) for better quality
)

# rotation + translation
data_aug_CPU_rotation_translation = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=10,            # Rotate images by up to 10 degrees
    width_shift_range=0.1,       # Horizontal shifts up to 10%
    height_shift_range=0.1,      # Vertical shifts up to 10%
    fill_mode='nearest',         # Fill strategy for pixels outside boundaries
    interpolation_order=1,       # Use bilinear interpolation (order=1) for better quality
)

# rotation + translation + flip
data_aug_CPU_rotation_translation_flip = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=10,            # Rotate images by up to 10 degrees
    width_shift_range=0.1,       # Horizontal shifts up to 10%
    height_shift_range=0.1,      # Vertical shifts up to 10%
    horizontal_flip=True,        # Random horizontal flips
    fill_mode='nearest',         # Fill strategy for pixels outside boundaries
    interpolation_order=1,       # Use bilinear interpolation (order=1) for better quality
)

# rotation + translation + flip + zoom
data_aug_CPU_rotation_translation_flip_zoom = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=10,            # Rotate images by up to 10 degrees
    width_shift_range=0.1,       # Horizontal shifts up to 10%
    height_shift_range=0.1,      # Vertical shifts up to 10%
    horizontal_flip=True,        # Random horizontal flips
    zoom_range=0.1,              # Random zoom up to 10%
    fill_mode='nearest',         # Fill strategy for pixels outside boundaries
    interpolation_order=1,       # Use bilinear interpolation (order=1) for better quality
)


Now we can select 2 images and apply sequentially the 4 augmentation techniques (rotation, translation, flip, zoom) using keras layers, and then apply the same techniques using ImageDataGenerator, and compare the results. We will use a seed of 128 during all comparisons and bilinear interpolation for all transformations, as well as nearest fill mode for all transformations.

In [ ]:
# visualize augmentation effects on 3 sample images
